In [1]:
import eval_common as _ec
_ec.set_axis("mechanism")     # change to "lob" for the line-of-business axis IGNORE
AXIS = _ec.AXIS
print("AXIS =", AXIS, "| categories:", _ec.MECH_ORDER)

AXIS = mechanism | categories: ['Reliability', 'Bias & Fairness', 'Privacy, Confidentiality & Infringement', 'Security & Misuse', 'Autonomous Actions', 'Governance, Oversight & Explainability']


In [2]:
from pathlib import Path
import numpy as np, pandas as pd
OUTPUT_DIR=Path("model_outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
DATE_COL="date"
import eval_common as _ec
MECH_ORDER = _ec.MECH_ORDER
df = _ec.load_incidents()
counts = _ec.build_counts(df)
print(counts.shape, counts.index[0], "->", counts.index[-1])

(127, 6) 2016-01 -> 2026-07


In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
evaluate = _ec.evaluate
mlog = _ec.mlog

In [4]:
rows=[]
for (yr, half) in _ec.TEST_FOLDS:
    tr, te = _ec.train_test_split_for(counts, yr, half)
    if tr.empty or te.empty: continue
    p = (tr.sum()/tr.values.sum()).reindex(MECH_ORDER)
    for m, a in te.iterrows():
        n = a.sum()
        if n <= 0: continue
        for k in MECH_ORDER:
            rows.append({"model":"static_historical_share",
                         "fold": _ec.fold_label(yr, half),
                         "month_period": str(m), "category": k,
                         "actual_count": float(a[k]),
                         "actual_share": float(a[k]/n),
                         "predicted_share": float(p[k])})
pred=pd.DataFrame(rows); overall, by_fold = evaluate(pred); display(overall); display(by_fold)
pred.to_csv(OUTPUT_DIR / f"predictions_static_{AXIS}.csv", index=False)
overall.to_csv(OUTPUT_DIR / f"metrics_static_overall_{AXIS}.csv", index=False)
by_fold.to_csv(OUTPUT_DIR / f"metrics_static_byfold_{AXIS}.csv", index=False)   # NEW: save per-fold too

,model,mae,rmse,mean_log_score_per_incident
0,static_historical_share,0.106155,0.140132,-1.689475


,model,fold,mae,rmse
0,static_historical_share,2020-H1,0.162582,0.207731
1,static_historical_share,2020-H2,0.109647,0.138862
2,static_historical_share,2021-H1,0.122210,0.144919
3,static_historical_share,2021-H2,0.115256,0.131852
4,static_historical_share,2022-H1,0.079881,0.095345
5,static_historical_share,2022-H2,0.101886,0.130084
6,static_historical_share,2023-H1,0.114217,0.158111
7,static_historical_share,2023-H2,0.110210,0.146414
8,static_historical_share,2024-H1,0.121398,0.164003
9,static_historical_share,2024-H2,0.108292,0.146380
